In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def generate_base_dataset(seed, n_samples=200):
    np.random.seed(seed)
    X_class1 = np.random.multivariate_normal(mean=[3, 3], cov=[[0.75, 0], [0, 0.75]], size=n_samples // 2)
    X_class2 = np.random.multivariate_normal(mean=[-3, -3], cov=[[0.75, 0], [0, 0.75]], size=n_samples // 2)
    X = np.vstack([X_class1, X_class2])
    y = np.array([1] * (n_samples // 2) + [-1] * (n_samples // 2))
    return X, y

def add_uniform_label_noise(y, noise_ratio=0.1):
    n_noise = int(len(y) * noise_ratio)
    noise_indices = np.random.choice(len(y), n_noise, replace=False)
    y_noisy = y.copy()
    y_noisy[noise_indices] *= -1
    return y_noisy

def add_gaussian_label_noise(X, y, noise_scale=1.0):
    y_noisy = y.copy()
    distances = np.linalg.norm(X, axis=1)
    prob_flip = 1 / (1 + np.exp(-distances / noise_scale))
    random_values = np.random.rand(len(y))
    flip_indices = random_values < prob_flip * 0.5
    y_noisy[flip_indices] *= -1
    return y_noisy

def add_feature_noise(X, noise_std=0.5):
    noise = np.random.normal(loc=0.0, scale=noise_std, size=X.shape)
    return X + noise

def add_outliers(X, y, n_outliers=5, outlier_distance=10):
    X_outliers = np.random.uniform(low=-outlier_distance, high=outlier_distance, size=(n_outliers, X.shape[1]))
    y_outliers = np.random.choice([-1, 1], size=n_outliers)
    X_augmented = np.vstack([X, X_outliers])
    y_augmented = np.concatenate([y, y_outliers])
    return X_augmented, y_augmented

datasets = {
    "clean": [],
    "clean_outliers": [],
    "uniform_target_noise": [],
    "gaussian_target_noise": [],
    "feature_noise": [],
    "uniform_target_noise_outliers": [],
    "gaussian_target_noise_outliers": [],
    "feature_noise_outliers": []
}


label_noise_ratio = 0.1
feature_noise_std = 0.75

for seed in range(5):
    X, y = generate_base_dataset(seed)

    y_uniform_noise = add_uniform_label_noise(y, noise_ratio=label_noise_ratio)
    y_gaussian_noise = add_gaussian_label_noise(X, y, noise_scale=3.0)
    X_feature_noise = add_feature_noise(X, noise_std=feature_noise_std)

    datasets["clean"].append((X.copy(), y.copy()))
    datasets["clean_outliers"].append(add_outliers(X.copy(), y.copy()))
    datasets["uniform_target_noise"].append((X.copy(), y_uniform_noise))
    datasets["gaussian_target_noise"].append((X.copy(), y_gaussian_noise))
    datasets["feature_noise"].append((X_feature_noise, y.copy()))
    datasets["uniform_target_noise_outliers"].append(add_outliers(X.copy(), y_uniform_noise))
    datasets["gaussian_target_noise_outliers"].append(add_outliers(X.copy(), y_gaussian_noise))
    datasets["feature_noise_outliers"].append(add_outliers(X_feature_noise.copy(), y.copy()))


print("All datasets generated including feature noise variations.")


All datasets generated including feature noise variations.


In [ ]:
def load_keel_file(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    data_start = False
    data = []
    for line in lines:
        line = line.strip()
        if line.lower().startswith("@data"):
            data_start = True
            continue
        if data_start and line:
            data.append(line.split(","))

    df = pd.DataFrame(data)

    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except ValueError:
            pass

    X = df.iloc[:, :-1].astype(float).values
    y_raw = df.iloc[:, -1].values

    unique_classes = np.unique(y_raw)
    if len(unique_classes) != 2:
        raise ValueError(f"Dataset {file_path} is not binary: classes = {unique_classes}")

    y = np.where(y_raw == unique_classes[0], -1, 1)

    return X, y


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

class ELMClassifier:
    def __init__(self, n_hidden=1000, activation='sigmoid', C=1.0, gamma=1.0, random_state=None):
        self.n_hidden = n_hidden
        self.activation = activation
        self.C = C
        self.gamma = gamma
        self.random_state = np.random.RandomState(random_state)
        self.scaler = StandardScaler()

    def _activation(self, X):
        if self.activation == 'sigmoid':
            Z = X @ self.a.T + self.b
            return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))
        elif self.activation == 'rbf':
            X_sq = np.sum(X**2, axis=1, keepdims=True)
            A_sq = np.sum(self.a**2, axis=1, keepdims=True).T
            dist_sq = X_sq - 2 * X @ self.a.T + A_sq
            return np.exp(-self.gamma * dist_sq)
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

    def fit(self, X, y):
        X = self.scaler.fit_transform(X)
        N, d = X.shape
        self.classes_ = np.unique(y)
        self.encoder = OneHotEncoder(sparse_output=False).fit(y.reshape(-1, 1))
        T = self.encoder.transform(y.reshape(-1, 1))

        self.a = self.random_state.normal(size=(self.n_hidden, d))
        self.b = None if self.activation == 'rbf' else self.random_state.normal(size=(1, self.n_hidden))

        H = self._activation(X)
        self.beta = np.linalg.solve(H.T @ H + np.eye(H.shape[1]) / self.C, H.T @ T)

    def predict(self, X):
        X = self.scaler.transform(X)
        H = self._activation(X)
        output = H @ self.beta
        return self.encoder.inverse_transform(output)

    def score(self, X, y):
        y_pred = self.predict(X)
        return accuracy_score(y, y_pred)

    def get_weight_vector(self):
        """
        Returns flattened output weight vector β
        """
        return self.beta.flatten()


In [ ]:
def angular_deviation(vec1, vec2):
    v1 = vec1 / np.linalg.norm(vec1)
    v2 = vec2 / np.linalg.norm(vec2)
    dot_product = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.arccos(dot_product) * (180.0 / np.pi)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

C_exponents = np.arange(-18, 19, 2)
C_values = [np.power(2.0, k) for k in C_exponents]
hidden_sizes = list(range(1000, 3001, 200))
results = {}

for n_hidden in hidden_sizes:
    for C in C_values:
        key = (n_hidden, C)
        results[key] = {}

        for dtype in datasets.keys():
            deviations = []
            for seed in range(5):
                X_clean, y_clean = datasets["clean"][seed]
                model_ref = ELMClassifier(n_hidden=n_hidden, C=C, random_state=0)
                model_ref.fit(X_clean, y_clean)
                beta_ref = model_ref.get_weight_vector()

                X, y = datasets[dtype][seed]

                model = ELMClassifier(n_hidden=n_hidden, C=C, random_state=0)
                model.fit(X, y)
                beta = model.get_weight_vector()
                deviation = angular_deviation(beta_ref, beta)

                deviations.append(deviation)

            results[key][dtype] = np.mean(deviations)

In [ ]:
best_params = {}

for dtype in datasets.keys():
    min_dev = float('inf')
    best_n = None
    best_C = None

    for n_hidden in hidden_sizes:
        for C in C_values:
            angle = results[(n_hidden, C)][dtype]
            if angle < min_dev:
                min_dev = angle
                best_n = n_hidden
                best_C = C

    best_params[dtype] = (best_n, best_C, min_dev)

# Tabulate results
headers = ["Dataset", "Best n_hidden", "Best C", "Min Deviation (°)"]
table = [[k, v[0], v[1], f"{v[2]:.2f}"] for k, v in best_params.items()]
print("\n=== Best Hyperparameters for Minimum Angular Deviation ===")
print(tabulate(table, headers=headers, tablefmt="grid"))



=== Best Hyperparameters for Minimum Angular Deviation ===
+--------------------------------+-----------------+-------------+---------------------+
| Dataset                        |   Best n_hidden |      Best C |   Min Deviation (°) |
+================================+=================+=============+=====================+
| clean                          |            1000 | 0.015625    |                0    |
+--------------------------------+-----------------+-------------+---------------------+
| clean_outliers                 |            2600 | 1.52588e-05 |                0.98 |
+--------------------------------+-----------------+-------------+---------------------+
| uniform_target_noise           |            2000 | 6.10352e-05 |                2.89 |
+--------------------------------+-----------------+-------------+---------------------+
| gaussian_target_noise          |            1600 | 6.10352e-05 |               14.98 |
+--------------------------------+----------------

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

C_exponents = np.arange(-18, 19, 2)
C_labels = [f"2^{k}" for k in C_exponents]

for dataset in datasets.keys():
    heatmap_data = []

    for n_hidden in hidden_sizes:
        row = []
        for C in C_values:
            angle = results[(n_hidden, C)][dataset]
            row.append(angle)
        heatmap_data.append(row)

    df = pd.DataFrame(heatmap_data, index=hidden_sizes, columns=C_labels)

    plt.figure(figsize=(12, 6))
    sns.heatmap(df, annot=True, fmt=".1f", cmap="coolwarm", cbar_kws={'label': 'Angular Deviation (°)'})
    plt.title(f"Angular Deviation Heatmap for {dataset.replace('_', ' ').title()}")
    plt.xlabel("C (Regularization Parameter)")
    plt.ylabel("Hidden Layer Size")
    plt.tight_layout()

    os.makedirs("heatmap_synthetic2", exist_ok=True)
    plt.savefig(f"heatmap_synthetic2/elm_heatmap_angular_deviation_{dataset}.png", dpi=300)
    plt.close()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tabulate import tabulate
from sklearn.model_selection import train_test_split

C_exponents = np.arange(-18, 19, 2)
C_values = [np.power(2.0, k) for k in C_exponents]
hidden_sizes = list(range(1000, 3001, 200))

dataset_paths = {
    "heart": {
        "standard": "/content/heart_standard.dat",
        "class_noise": "/content/heart_class_noise.dat",
        "attribute_noise": "/content/heart_attribute_noise.dat"
    },
    "pima": {
        "standard": "/content/pima_standard.dat",
        "class_noise": "/content/pima_class_noise.dat",
        "attribute_noise": "/content/pima_attribute_noise.dat"
    },
    "ionosphere": {
        "standard": "/content/ionosphere_standard.dat",
        "class_noise": "/content/ionosphere_class_noise.dat",
        "attribute_noise": "/content/ionosphere_attribute_noise.dat"
    },
    "sonar": {
        "standard": "/content/sonar_standard.dat",
        "class_noise": "/content/sonar_class_noise.dat",
        "attribute_noise": "/content/sonar_attribute_noise.dat"
    }
}


results_class_noise = {}
results_attribute_noise = {}

for n_hidden in hidden_sizes:
    for C in C_values:
        key = (n_hidden, C)
        results_class_noise[key] = {}
        results_attribute_noise[key] = {}

        for name, paths in dataset_paths.items():
            X_std, y_std = load_keel_file(paths["standard"])
            X_cls, y_cls = load_keel_file(paths["class_noise"])
            X_att, y_att = load_keel_file(paths["attribute_noise"])

            model_std = ELMClassifier(n_hidden=n_hidden, C=C, random_state=0)
            model_std.fit(X_std, y_std)
            beta_std = model_std.get_weight_vector()

            model_cls = ELMClassifier(n_hidden=n_hidden, C=C, random_state=0)
            model_cls.fit(X_cls, y_cls)
            beta_cls = model_cls.get_weight_vector()

            model_att = ELMClassifier(n_hidden=n_hidden, C=C, random_state=0)
            model_att.fit(X_att, y_att)
            beta_att = model_att.get_weight_vector()

            results_class_noise[key][name] = angular_deviation(beta_std, beta_cls)
            results_attribute_noise[key][name] = angular_deviation(beta_std, beta_att)




In [ ]:
best_params_class = {}

for name in dataset_paths:
    min_dev = float('inf')
    best_C, best_n = None, None

    for n_hidden in hidden_sizes:
        for C in C_values:
            angle = results_class_noise[(n_hidden, C)][name]
            if angle < min_dev:
                min_dev = angle
                best_C = C
                best_n = n_hidden

    best_params_class[name] = (best_n, best_C, min_dev)

print("\n=== Best ELM Hyperparameters (Standard vs Class Noise) ===")
headers = ["Dataset", "Best n_hidden", "Best C", "Min Angular Deviation (°)"]
table = [[k, v[0], v[1], f"{v[2]:.2f}"] for k, v in best_params_class.items()]
print(tabulate(table, headers=headers, tablefmt="grid"))



=== Best ELM Hyperparameters (Standard vs Class Noise) ===
+------------+-----------------+-------------+-----------------------------+
| Dataset    |   Best n_hidden |      Best C |   Min Angular Deviation (°) |
+============+=================+=============+=============================+
| heart      |            1000 | 6.10352e-05 |                           0 |
+------------+-----------------+-------------+-----------------------------+
| pima       |            1000 | 1.52588e-05 |                           0 |
+------------+-----------------+-------------+-----------------------------+
| ionosphere |            1000 | 3.8147e-06  |                           0 |
+------------+-----------------+-------------+-----------------------------+
| sonar      |            1000 | 6.10352e-05 |                           0 |
+------------+-----------------+-------------+-----------------------------+


In [ ]:
best_params_attribute = {}

for name in dataset_paths:
    min_dev = float('inf')
    best_C, best_n = None, None

    for n_hidden in hidden_sizes:
        for C in C_values:
            angle = results_attribute_noise[(n_hidden, C)][name]
            if angle < min_dev:
                min_dev = angle
                best_C = C
                best_n = n_hidden

    best_params_attribute[name] = (best_n, best_C, min_dev)

print("\n=== Best ELM Hyperparameters (Standard vs Attribute Noise) ===")
table = [[k, v[0], v[1], f"{v[2]:.2f}"] for k, v in best_params_attribute.items()]
print(tabulate(table, headers=headers, tablefmt="grid"))



=== Best ELM Hyperparameters (Standard vs Attribute Noise) ===
+------------+-----------------+------------+-----------------------------+
| Dataset    |   Best n_hidden |     Best C |   Min Angular Deviation (°) |
+============+=================+============+=============================+
| heart      |            1000 | 3.8147e-06 |                        1.65 |
+------------+-----------------+------------+-----------------------------+
| pima       |            1000 | 3.8147e-06 |                        1.58 |
+------------+-----------------+------------+-----------------------------+
| ionosphere |            1000 | 3.8147e-06 |                        2.5  |
+------------+-----------------+------------+-----------------------------+
| sonar      |            1000 | 3.8147e-06 |                        3.31 |
+------------+-----------------+------------+-----------------------------+


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

C_exponents = np.arange(-18, 19, 2)
C_labels = [f"2^{k}" for k in C_exponents]

os.makedirs("heatmap_keel_class_noise", exist_ok=True)

for name in dataset_paths:
    heatmap_data = []

    for n_hidden in hidden_sizes:
        row = []
        for C in C_values:
            row.append(results_class_noise[(n_hidden, C)][name])
        heatmap_data.append(row)

    df = pd.DataFrame(heatmap_data, index=hidden_sizes, columns=C_labels)

    plt.figure(figsize=(12, 6))
    sns.heatmap(df, annot=True, fmt=".1f",
                cmap="coolwarm",
                cbar_kws={'label': 'Angular Deviation (°)'})
    plt.title(f"ELM Angular Deviation: Standard vs Class Noise ({name.title()})")
    plt.xlabel("C Value (log scale)")
    plt.ylabel("Hidden Layer Size")
    plt.tight_layout()
    plt.savefig(f"heatmap_keel_class_noise/elm_class_noise_{name}.png", dpi=300)
    plt.close()



In [ ]:
os.makedirs("heatmap_keel_attribute_noise", exist_ok=True)

for name in dataset_paths:
    heatmap_data = []

    for n_hidden in hidden_sizes:
        row = []
        for C in C_values:
            row.append(results_attribute_noise[(n_hidden, C)][name])
        heatmap_data.append(row)

    df = pd.DataFrame(heatmap_data, index=hidden_sizes, columns=C_labels)

    plt.figure(figsize=(12, 6))
    sns.heatmap(df, annot=True, fmt=".1f",
                cmap="coolwarm",
                cbar_kws={'label': 'Angular Deviation (°)'})
    plt.title(f"ELM Angular Deviation: Standard vs Attribute Noise ({name.title()})")
    plt.xlabel("C Value (log scale)")
    plt.ylabel("Hidden Layer Size")
    plt.tight_layout()
    plt.savefig(f"heatmap_keel_attribute_noise/elm_attribute_noise_{name}.png", dpi=300)
    plt.close()
